#### Model Training

In [36]:
import numpy as np
import pandas as pd
import joblib
import data_ingestion
import pipeline
import train_model
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report

import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Importing

In [37]:
query = """
    SELECT review_text, sentiment_label
    FROM reviews;
"""
conn = data_ingestion.get_connection()

In [38]:
df = pd.read_sql_query(query, conn)
df

,review_text,sentiment_label
0,Barang sudah diterima dengan cepat dan kualita...,positive
1,alat yg sangat bermanfaat untuk melatih skill ...,positive
2,mantap barang sudah diterima .. sesuai dgn yg ...,positive
3,Good product good service. Thanks,positive
4,transfer tgl 17 barang nya nyampe bali tgl 18....,positive
...,...,...
65538,Mantapp,positive
65539,"Kualitas produk: Kualitas produknya bagus, bua...",positive
65540,packing rapih. produk aman sampai rumah. berfu...,positive
65541,Bagus,positive


Data Cleaning (check train_model for detailed funcs)

In [39]:
train_model.clean_df(df)

Checking for any missing value in columns..
No missing value in column 0
No missing value in column 1
Following columns have missing value present: ['None']
-----------------------
Checking for any duplicate rows..
A total of 7460 duplicate rows have been found in the dataset. Removing them..
0 duplicate rows present in the database.
58083 rows left in the database.
-----------------------


,review_text,sentiment_label
0,Barang sudah diterima dengan cepat dan kualita...,positive
1,alat yg sangat bermanfaat untuk melatih skill ...,positive
2,mantap barang sudah diterima .. sesuai dgn yg ...,positive
3,Good product good service. Thanks,positive
4,transfer tgl 17 barang nya nyampe bali tgl 18....,positive
...,...,...
65535,"packing rapi, respon n proses kirim cpt.. Reco...",positive
65536,kualitas original dan pemakaian nyaman,positive
65539,"Kualitas produk: Kualitas produknya bagus, bua...",positive
65540,packing rapih. produk aman sampai rumah. berfu...,positive


In [40]:
slang_dict = pipeline.initialize_dict("colloquial-indonesian-lexicon.csv")
df["review_text"] = df["review_text"].apply(lambda x : pipeline.clean_review_text(text=x, data_dict=slang_dict))
df.head(5)

,review_text,sentiment_label
0,barang sudah diterima dengan cepat dan kualita...,positive
1,alat yang sangat bermanfaat untuk melatih skil...,positive
2,mantap barang sudah diterima sesuai dengan yan...,positive
3,good product good service thanks,positive
4,transfer tanggal 17 barang nya sampai bali tan...,positive


In [41]:
df["sentiment_label"] = df["sentiment_label"].map({"negative" : 0, "neutral" : 1, "positive" : 2})

In [42]:
df[["sentiment_label"]].value_counts()

sentiment_label
2                  56506
0                    790
1                    787
Name: count, dtype: int64

Train-Test split

In [43]:
X = df["review_text"]
y = df["sentiment_label"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True, stratify=y)

Text Vectorizing

In [44]:
indo_stopwords = stopwords.words("indonesian")

In [45]:
vectorizer = TfidfVectorizer(stop_words=indo_stopwords, ngram_range=(1, 2), min_df=3, max_features=15000) # indonesian stop words, unigram and bigram range, minimum document frequency = 3 to ignore one time typo usually, only take into account the top 15000 most impactful words basically

In [46]:
X_train_tfidf = vectorizer.fit_transform(X_train)

c:\Users\User\anaconda3\envs\ecom_nlp_env\Lib\site-packages\sklearn\feature_extraction\text.py:411: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['baiknya', 'berkali', 'kali', 'kurangnya', 'mata', 'olah', 'sekurang', 'setidak', 'tama', 'tidaknya'] not in stop_words.
  warnings.warn(


In [47]:
X_test_tfidf = vectorizer.transform(X_test)

Training

In [48]:
lr_model = LogisticRegression(penalty="l2", class_weight="balanced", C=1.0, max_iter=1000, random_state=42) # l2 (ridge regularization) is the default, class weight balanced is needed since our dependent variable is very unbalanced, regularization strength to 1 since our dataset is very sparse (15000 features is a lot) so C ensures that the model dont overfit (also higher C score reduces the strength), max iter lower than 1000 gives ConvergenceWarning because not enough training iteration for the model to reach optimal convergence when tested
linearsvc_model = LinearSVC(penalty="l2", class_weight="balanced", C=1.0, dual=False, multi_class="ovr", random_state=42) # penalty and class weight follows our LR model. Select the algorithm to either solve the dual or primal optimization problem. Prefer dual=False when n_samples > n_features (from sklearn documentation). our n_samples is around 58k while our n_features is exactly 15k, so dual will be False. ovr is the standard, its also better for text based prediction
# random state for defensive programming practice (reproducibility)

In [49]:
lr_model.fit(X_train_tfidf, y_train)

c:\Users\User\anaconda3\envs\ecom_nlp_env\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'l2'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multicl

In [50]:
linearsvc_model.fit(X_train_tfidf, y_train)

,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower the impact of regularization on it.Then, the weights become `[w_x_1, ..., w_x_n,w_intercept*intercept_scaling]`, where `w_x_1, ..., w_x_n` representthe feature weights and the intercept weight is scaled by`intercept_scaling`. This scaling allows the intercept term to have adifferent regularization behavior compared to the other features.",1
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to ``class_weight[i]*C`` forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",'balanced'
,"verbose verbose: int, default=0Enable verbose output. Note that this setting takes advantage of aper-process runtime setting in liblinear that, if enabled, may not workproperly in a multithreaded context.",0
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo

Predictions

In [51]:
lr_preds = lr_model.predict(X_test_tfidf)
linearsvc_preds = linearsvc_model.predict(X_test_tfidf)

Evaluation

In [52]:
print("---Logistic Regression---")
print(classification_report(y_test, lr_preds))
print("---Linear SVC---")
print(classification_report(y_test, linearsvc_preds))

---Logistic Regression---
              precision    recall  f1-score   support

           0       0.16      0.43      0.24       158
           1       0.09      0.31      0.14       157
           2       0.99      0.93      0.96     11302

    accuracy                           0.92     11617
   macro avg       0.41      0.56      0.44     11617
weighted avg       0.97      0.92      0.94     11617

---Linear SVC---
              precision    recall  f1-score   support

           0       0.33      0.32      0.32       158
           1       0.16      0.15      0.16       157
           2       0.98      0.98      0.98     11302

    accuracy                           0.96     11617
   macro avg       0.49      0.48      0.49     11617
weighted avg       0.96      0.96      0.96     11617



Precision: Out of all the instances model predicted positive, how many are correct?
Recall (Sensitivity): Out of all actual positives, how many did the model find?
F1-Score is the harmonic mean score of the two.

Precision focuses more on minimizing False Positives.
Recall focuses more on minimizing False Negatives.

Since both precision and recall are equally important here, we can look at our F1-Score. We know our dataset is heavily imbalanced, so judging through our macro avg score, we know that LinearSVC performed better here.

In [55]:
joblib.dump(lr_model, "models/sentiment_lr_model.pkl")
joblib.dump(linearsvc_model, "models/sentiment_linearsvc_model.pkl")
joblib.dump(vectorizer, "models/sentiment_vectorizer.pkl") 

['models/sentiment_vectorizer.pkl']